In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# データ操作と数値計算
import pandas as pd
import numpy as np
import re # 正規表現用

# 自然言語処理 (NLP) 関連
import nltk
from nltk.corpus import stopwords # ストップワード除去用
from nltk.tokenize import word_tokenize # 単語分割用
# Hugging Face Transformers と PyTorch
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertForSequenceClassification, BertTokenizer
from torch.optim import AdamW # AdamWはtorch.optimから直接インポート
from transformers import get_linear_schedule_with_warmup

# モデル評価とデータ分割
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

# 可視化ライブラリ
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# その他のユーティリティ
import os # ファイルパス操作用

In [ ]:
print("--- データの読み込みと初期確認 ---")
# データセットのパスを定義
train_path = '/kaggle/input/nlp-getting-started/train.csv'
test_path = '/kaggle/input/nlp-getting-started/test.csv'
sample_submission_path = '/kaggle/input/nlp-getting-started/sample_submission.csv'

# CSVファイルをPandas DataFrameとして読み込む
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission_df = pd.read_csv(sample_submission_path)

# データの形状と最初の数行を表示して確認
print("\n訓練データ (train_df) の情報:")
print(train_df.shape)
print(train_df.head())

print("\nテストデータ (test_df) の情報:")
print(test_df.shape)
print(test_df.head())

# 欠損値の確認
print("\n訓練データの欠損値:")
print(train_df.isnull().sum())
print("\nテストデータの欠損値:")
print(test_df.isnull().sum())

In [ ]:
print("\n--- 欠損値の処理 ---")
# 'keyword'と'location'カラムの欠損値を'no_keyword'/'no_location'で埋める
# BERTはテキストの文脈を重視するため、これらの情報は直接利用しないが、データの一貫性を保つため処理
train_df['keyword'] = train_df['keyword'].fillna('no_keyword')
test_df['keyword'] = test_df['keyword'].fillna('no_keyword')
train_df['location'] = train_df['location'].fillna('no_location')
test_df['location'] = test_df['location'].fillna('no_location')

In [ ]:
print("\n--- BERT向けテキストクリーニング ---")
# BERTは前処理に非常に強いが、URLやHTMLタグ、絵文字は意味を歪める可能性があるため除去
def clean_text_for_bert(text):
    # URLを除去
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # HTMLタグを除去
    text = re.sub(r'<.*?>', '', text)
    # 絵文字を除去
    emoji_pattern = re.compile(
        '['
        '\U0001F600-\U0001F64F' '\U0001F300-\U0001F5FF'
        '\U0001F680-\U0001F6FF' '\U0001F1E0-\U0001F1FF'
        '\U00002702-\U000027B0' '\U000024C2-\U0001F251'
        ']+',
        flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)
    # BERTは小文字化を内部で行うため、ここでは明示的に行わない
    # BERTは句読点や数字もトークンとして扱うので、ここでは除去しない
    return text

# クリーニング関数を適用
train_df['cleaned_text_bert'] = train_df['text'].apply(clean_text_for_bert)
test_df['cleaned_text_bert'] = test_df['text'].apply(clean_text_for_bert)

In [ ]:
# ターゲット変数とクリーニング済みテキストデータを取得
sentences = train_df['cleaned_text_bert'].values
labels = train_df['target'].values
test_sentences = test_df['cleaned_text_bert'].values

print(f"\n訓練データのツイート数: {len(sentences)}")
print(f"テストデータのツイート数: {len(test_sentences)}")
print(f"最初の訓練ツイートの例: {sentences[0]}")

In [ ]:
print("\n--- 仮説検証のための可視化を開始します ---")

# ターゲット変数ごとのキーワード出現頻度
plt.figure(figsize=(14, 7))
sns.countplot(data=train_df[train_df['keyword'] != 'no_keyword'], y='keyword', hue='target',
              order=train_df[train_df['keyword'] != 'no_keyword']['keyword'].value_counts().index[:20])
plt.title('Keyword Frequency by Target (Top 20)')
plt.xlabel('Frequency')
plt.ylabel('Keyword')
plt.tight_layout()
plt.show()

# ワードクラウドの生成
stop_words = set(stopwords.words('english'))

def generate_wordcloud(text_data, title):
    # テキストを結合し、小文字化、単語分割、ストップワード除去
    all_words = ' '.join(text_data).lower()
    words = word_tokenize(all_words)
    filtered_words = [word for word in words if word.isalnum() and word not in stop_words]
    cleaned_text = ' '.join(filtered_words)

    wordcloud = WordCloud(width=800, height=400, background_color='white',
                          collocations=False).generate(cleaned_text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    plt.show()

# 災害ツイートのワードクラウド
generate_wordcloud(train_df[train_df['target'] == 1]['cleaned_text_bert'], 'Word Cloud for Disaster Tweets')

# 非災害ツイートのワードクラウド
generate_wordcloud(train_df[train_df['target'] == 0]['cleaned_text_bert'], 'Word Cloud for Non-Disaster Tweets')


# ターゲット変数ごとのロケーション分布 (上位20)
plt.figure(figsize=(14, 7))
sns.countplot(data=train_df[train_df['location'] != 'no_location'], y='location', hue='target',
              order=train_df[train_df['location'] != 'no_location']['location'].value_counts().index[:20])
plt.title('Location Frequency by Target (Top 20)')
plt.xlabel('Frequency')
plt.ylabel('Location')
plt.tight_layout()
plt.show()

# 欠損値の分布 (keywordとlocation)
# 欠損値の有無を示す新しいカラムを作成
train_df['keyword_missing'] = train_df['keyword'] == 'no_keyword'
train_df['location_missing'] = train_df['location'] == 'no_location'

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.countplot(data=train_df, x='keyword_missing', hue='target')
plt.title('Relationship between Keyword Missingness and Target')
plt.xlabel('Keyword Missing (True: Missing)')
plt.ylabel('Count')
plt.xticks(ticks=[0, 1], labels=['False', 'True'])

plt.subplot(1, 2, 2)
sns.countplot(data=train_df, x='location_missing', hue='target')
plt.title('Relationship between Location Missingness and Target')
plt.xlabel('Location Missing (True: Missing)')
plt.ylabel('Count')
plt.xticks(ticks=[0, 1], labels=['False', 'True'])

plt.tight_layout()
plt.show()

print("\n--- 仮説検証のための可視化が完了しました ---")

In [ ]:
print("\n--- BERTトークナイザーのロードとテキストのエンコード ---")
# 'bert-base-uncased'モデルのトークナイザーをロード
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

# テキストをトークン化し、BERT入力形式に変換する関数
def encode_texts(tokenizer, texts, max_len=128):
    input_ids = []
    attention_masks = []

    for text in texts:
        encoded_dict = tokenizer.encode_plus(
                            text,
                            add_special_tokens = True,  # [CLS]と[SEP]トークンを追加
                            max_length = max_len,       # 最大長を設定
                            padding = 'max_length',     # 最大長までパディング
                            truncation = True,          # 最大長を超える場合は切り捨て
                            return_attention_mask = True, # アテンションマスクを返す
                            return_tensors = 'pt',      # PyTorchテンソルとして返す
                       )

        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])

    # リスト内のテンソルを結合
    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)

    return input_ids, attention_masks

# 訓練データとテストデータをBERT形式に変換
MAX_LEN = 128 # BERTモデルの入力最大長
train_inputs, train_masks = encode_texts(tokenizer, sentences, MAX_LEN)
test_inputs, test_masks = encode_texts(tokenizer, test_sentences, MAX_LEN)

# ラベルをPyTorchテンソルに変換
train_labels = torch.tensor(labels)

print(f"エンコードされた訓練データの入力ID形状: {train_inputs.shape}")
print(f"エンコードされた訓練データのアテンションマスク形状: {train_masks.shape}")
print(f"訓練ラベルの形状: {train_labels.shape}")

print("\n--- 訓練データと検証データの分割 ---")
# 訓練データセットを訓練用と検証用に分割 (10%を検証データに)
train_inputs, validation_inputs, train_masks, validation_masks, train_labels, validation_labels = train_test_split(
    train_inputs, train_masks, train_labels, random_state=42, test_size=0.1
)

print(f"分割後の訓練データセットの形状: {train_inputs.shape}")
print(f"分割後の検証データセットの形状: {validation_inputs.shape}")

print("\n--- PyTorch DataLoaderの作成 ---")
# PyTorchのDataLoaderを作成し、バッチ処理を効率化
batch_size = 32

# 訓練データ用
train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_sampler = RandomSampler(train_data) # ランダムにサンプリング
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

# 検証データ用
validation_data = TensorDataset(validation_inputs, validation_masks, validation_labels)
validation_sampler = SequentialSampler(validation_data) # 順次サンプリング
validation_dataloader = DataLoader(validation_data, sampler=validation_sampler, batch_size=batch_size)

# テストデータ用 (ラベルがないため入力とマスクのみ)
test_data = TensorDataset(test_inputs, test_masks)
test_sampler = SequentialSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)

In [ ]:
print("\n--- BERTモデルの準備 ---")
# GPUが利用可能か確認し、デバイスを設定
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('GPUが利用可能です:', torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print('GPUは利用できません。CPUを使用します。')

# BERTモデルをロード (シーケンス分類用)
# num_labels=2: 災害/非災害の2クラス分類
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", # 事前学習済みモデル
    num_labels = 2,      # 出力クラス数 (0:非災害, 1:災害)
    output_attentions = False, # アテンション重みは出力しない (計算コスト削減)
    output_hidden_states = False, # 隠れ状態は出力しない (計算コスト削減)
)

# モデルをGPUに移動 (GPUが利用可能な場合)
model.to(device)

# オプティマイザの設定
# BERT推奨のAdamWオプティマイザを使用
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8) # 学習率とイプシロン

# エポック数 (通常2-4エポックで十分なことが多い。長すぎると過学習のリスクあり)
epochs = 3

# 学習率スケジューラの設定 (ウォームアップ付き線形スケジューラ)
# 全訓練ステップ数を計算
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# 評価関数 (Accuracy)
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return accuracy_score(labels_flat, pred_flat)

print("\n--- BERTモデルのファインチューニングを開始します ---")
# モデル学習ループ
for epoch_i in range(0, epochs):
    print(f"\n======== エポック {epoch_i + 1} / {epochs} ========")
    print("訓練中...")

    model.train() # モデルを訓練モードに設定 (ドロップアウトなどが有効になる)

    total_loss = 0 # エポックごとの合計ロス

    # 訓練データローダーからバッチごとにデータを取得
    for step, batch in enumerate(train_dataloader):
        # バッチデータをGPUに移動
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        model.zero_grad() # 勾配をゼロクリア (前回のバッチの勾配が残らないように)

        # 順伝播: モデルにデータを入力し、出力とロスを計算
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask,
                        labels=b_labels)
        loss = outputs.loss # ロスを取得
        total_loss += loss.item() # ロスを加算

        # 逆伝播: ロスに基づいて勾配を計算
        loss.backward()
        # 勾配クリッピング: 勾配爆発を防ぐために勾配のL2ノルムを1.0にクリップ
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step() # オプティマイザでパラメータを更新
        scheduler.step() # 学習率を更新

        # 200ステップごとに進捗表示
        if step % 200 == 0 and not step == 0:
            print(f"  Batch {step:>5,}  Loss: {loss.item():.2f}")

    # エポックごとの平均訓練ロスを計算
    avg_train_loss = total_loss / len(train_dataloader)
    print(f"  平均訓練ロス: {avg_train_loss:.2f}")

    # --- 検証フェーズ ---
    print("\n検証中...")
    model.eval() # モデルを評価モードに設定 (ドロップアウトなどが無効になる)

    total_eval_accuracy = 0
    total_eval_f1 = 0
    nb_eval_steps = 0

    # 検証データローダーからバッチごとにデータを取得
    for batch in validation_dataloader:
        # バッチデータをGPUに移動
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        # 勾配計算を無効化 (メモリ使用量削減と計算速度向上)
        with torch.no_grad():
            outputs = model(b_input_ids,
                            token_type_ids=None,
                            attention_mask=b_input_mask,
                            labels=b_labels)

        logits = outputs.logits # モデルの出力 (ロジット)
        # ロジットとラベルをCPUに移動し、Numpy配列に変換
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        # 評価指標を計算し加算
        total_eval_accuracy += flat_accuracy(logits, label_ids)
        total_eval_f1 += f1_score(label_ids, np.argmax(logits, axis=1).flatten())

    # 検証フェーズの平均評価指標を計算
    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    avg_val_f1 = total_eval_f1 / len(validation_dataloader)
    print(f"  検証Accuracy: {avg_val_accuracy:.2f}")
    print(f"  検証F1スコア: {avg_val_f1:.2f}")

print("\n--- ファインチューニングが完了しました ---")

In [ ]:
print("\n--- テストデータでの予測を開始します ---")
model.eval() # モデルを評価モードに設定

predictions = [] # 予測結果を格納するリスト

# テストデータローダーからバッチごとにデータを取得
for batch in test_dataloader:
    # バッチデータをGPUに移動
    b_input_ids = batch[0].to(device)
    b_input_mask = batch[1].to(device)

    # 勾配計算を無効化
    with torch.no_grad():
        # モデルにデータを入力 (ラベルは不要)
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask)

    logits = outputs.logits # モデルの出力 (ロジット)
    # ロジットをCPUに移動し、Numpy配列に変換
    logits = logits.detach().cpu().numpy()
    predictions.append(logits)

# 予測結果を連結し、最も高いlogitを持つクラスを選択
# np.concatenateでリスト内のNumpy配列を結合
# np.argmaxで各行（ツイート）で最も高い値を持つインデックス（クラスID）を取得
final_predictions_bert = np.argmax(np.concatenate(predictions, axis=0), axis=1)

print(f"最終予測結果の形状: {final_predictions_bert.shape}")
print(f"最終予測結果の最初の5件: {final_predictions_bert[:5]}")

print("\n--- 提出用ファイルの作成と保存 ---")
# 提出用DataFrameを作成
submission_df_bert = sample_submission_df.copy()
submission_df_bert['target'] = final_predictions_bert # 予測結果を'target'カラムに代入

# 提出ファイルの最初の数行を表示して確認
print("\n生成された提出ファイルのプレビュー:")
print(submission_df_bert.head())

# 提出ファイルをCSV形式で保存
# index=Falseとすることで、DataFrameのインデックスがCSVに書き込まれないようにする
submission_df_bert.to_csv('submission.csv', index=False)

print("\nsubmission.csv が正常に生成され、/kaggle/working/ ディレクトリに保存されました。")
print("このファイルをKaggleに提出してください。")